# Deep Learning Lab 0

## Imports

In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
from torch.utils.data import random_split

import wandb
import os

## Wandb Setup


In [2]:
wandb.init(project="lab0")
#wandb.watch()


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Rasmus\_netrc.
wandb: Currently logged in as: rasmoose (rasmoose-lule-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


ServicePollForTokenError: Failed to read port info after 30.0 seconds.

## Preprocessing
### Checklist Train Data
- Normalization
- Data Augmentation
- Create images --> ToTensor()
- Dataloader
### Checklist Test Data
- Normalization
- Create images --> ToTensor()

In [3]:
"""
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4), # Randomly Crops 32x32 Region After Padding To Create Small Shifts --> Robustness
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), 
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
    ])

"""

transformer = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
    ])

transformer_MNIST = transforms.Compose([
    transforms.ToTensor(), 
    transforms.Normalize(mean=(0.5), std=(0.5))
    ])

transformer_SVHN = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5), std=(0.5))
])

traindata = torchvision.datasets.CIFAR10(
    root='./data', 
    train=True, 
    download=True, 
    transform=transformer)

testdata = torchvision.datasets.CIFAR10(
    root='./data', 
    train=False, 
    download=True, 
    transform=transformer)

100.0%
d:\Users\Rasmus\Anaconda3\envs\ADL\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [4]:
train_size = int(0.8*len(traindata))
val_size = len(traindata) - train_size

train_dataset, val_dataset = random_split(traindata, [train_size, val_size], generator=torch.Generator().manual_seed(42))

trainload = torch.utils.data.DataLoader(traindata, batch_size=32, shuffle=True)
valload = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)
testload = torch.utils.data.DataLoader(testdata, batch_size=32, shuffle=False)

## Training and Testing 
### Checklist
- Activation Function LeakyReLU
- Optimizer Stochastic Gradient Descent lr = 0.0001
- Accuracy on Test set
- Val/Train Stopping when the Val loss converges
- Tensorboard and wandb
- Schedulers


In [5]:
if torch.cuda.is_available():
    device = "cuda" #Nvidia Graphics Card
elif torch.backends.mps.is_available():
    device = "mps" # Apple
else:
    device = "cpu" # Worst Case
print(device)


cuda


In [14]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # Images: 32x32
            nn.BatchNorm2d(32),
            nn.LeakyReLU(),
            nn.MaxPool2d(2,2), # Images: (32x32)/(2*2) --> 16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(),
            nn.MaxPool2d(2,2) # Images  (16x16)/(2x2) --> 8x8
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 128), # 4096 --> 128
            nn.LeakyReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

class CNN_MNIST(nn.Module):
    def __init__(self, num_classes=10):
        super(CNN_MNIST, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), # Images: 32x32
            nn.BatchNorm2d(32),
            nn.LeakyReLU(),
            nn.MaxPool2d(2,2), # Images: (32x32)/(2*2) --> 16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(),
            nn.MaxPool2d(2,2) # Images  (16x16)/(2x2) --> 8x8
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 128), # 4096 --> 128
            nn.LeakyReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [9]:
model = CNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)
num_epochs = 10

In [10]:
# ------------------------- Wandb -------------------------


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode="min",
        factor = 0.1, 
        patience = 3 # Wait x amount of epochs before reducing lr
    )

best_val_loss = float('inf')

for epoch in range(num_epochs):
    # ------------------------- TRAIN -------------------------
    model.train()
    running_loss = 0.0

    for images, labels in trainload:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(trainload)

    # wandb.log({
    #     "loss": loss
    # })
    # ------------------------- VALIDATION -------------------------

    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in valload:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()

        #     wandb.log({
        #     "validation loss": val_running_loss
        # })

        average_val_loss = val_running_loss / len(valload)

        # wandb.log({
        #     "average validation loss": average_val_loss
        # })

    scheduler.step(average_loss)

    # ------------------------- Saving Best Model -------------------------
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        torch.save(model.state_dict(), "BestModel.pth")
        print("Saved The Best Performing Model")

    # ------------------------- Visualization -------------------------
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"train loss: {average_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )

Saved The Best Performing Model
Epoch [1/10] train loss: 2.197, val loss: 2.071, lr: 0.000100
Saved The Best Performing Model
Epoch [2/10] train loss: 2.003, val loss: 1.909, lr: 0.000100
Saved The Best Performing Model
Epoch [3/10] train loss: 1.879, val loss: 1.801, lr: 0.000100
Saved The Best Performing Model
Epoch [4/10] train loss: 1.793, val loss: 1.727, lr: 0.000100
Saved The Best Performing Model
Epoch [5/10] train loss: 1.729, val loss: 1.664, lr: 0.000100
Saved The Best Performing Model
Epoch [6/10] train loss: 1.678, val loss: 1.618, lr: 0.000100
Saved The Best Performing Model
Epoch [7/10] train loss: 1.637, val loss: 1.579, lr: 0.000100
Saved The Best Performing Model
Epoch [8/10] train loss: 1.600, val loss: 1.546, lr: 0.000100
Saved The Best Performing Model
Epoch [9/10] train loss: 1.570, val loss: 1.509, lr: 0.000100
Saved The Best Performing Model
Epoch [10/10] train loss: 1.545, val loss: 1.487, lr: 0.000100


In [11]:
model.load_state_dict(torch.load("BestModel.pth"))
model.to(device)
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # wandb.log({
        #     "test loss" : test_loss

        # })          

average_test_loss = test_loss / len(testload)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")

# wandb.finish()


Test loss: 1.495
Test accuracy: 48.66%


### Checklist 2
- LeakyReLU --> Tanh
- Optimizer: SGD --> Adam
- Visualize the results on a Tensorboard

In [12]:
epochs = 10
learning_rate = 0.0001

transformer = transforms.Compose([
    transforms.Resize((224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
    ])  

# Download CIFAR-10 datasets
trainset_full = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transformer)
testset_full = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transformer)

# Combine train and test, then split into 70/15/15
combined_dataset = torch.utils.data.ConcatDataset([trainset_full, testset_full])
total_size = len(combined_dataset)

train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

trainset, valset, testset = random_split(
    combined_dataset, 
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
valloader = torch.utils.data.DataLoader(valset, batch_size=64, shuffle=False)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

print(f"Dataset Split - Train: {train_size}, Val: {val_size}, Test: {test_size}")


print("\n" + "=" * 50)
print("EXPERIMENT 1: Fine-Tuning Pretrained AlexNet")
print("=" * 50)

model_finetune = models.alexnet(pretrained=True)
# Replace the last fully connected layer to output 10 classes (CIFAR-10)
model_finetune.classifier[6] = nn.Linear(4096, 10)
model_finetune = model_finetune.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_finetune = torch.optim.Adam(model_finetune.parameters(), lr=learning_rate)

# Training loop for fine-tuning
best_val_loss = float('inf')
for epoch in range(epochs):
    model_finetune.train()
    running_loss = 0.0
    
    for images, labels in trainloader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer_finetune.zero_grad()
        
        outputs = model_finetune(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer_finetune.step()
        
        running_loss += loss.item()
    
    avg_train_loss = running_loss / len(trainloader)
    
    # Validation
    model_finetune.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in valloader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model_finetune(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(valloader)
    
    print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model_finetune.state_dict(), "BestModel_Finetune.pth")

# Test the fine-tuned model
model_finetune.load_state_dict(torch.load("BestModel_Finetune.pth"))
model_finetune.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model_finetune(images)
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_finetune = 100 * (correct / total)
print(f"\nTest Accuracy (Fine-tuning): {accuracy_finetune:.2f}%")
# wandb.log({"finetune_test_accuracy": accuracy_finetune})

Dataset Split - Train: 42000, Val: 9000, Test: 9000

EXPERIMENT 1: Fine-Tuning Pretrained AlexNet


NameError: name 'models' is not defined

In [ ]:
# ======================== EXPERIMENT 2: Feature Extraction (Frozen Pretrained AlexNet) ========================
print("\n" + "=" * 50)
print("EXPERIMENT 2: Feature Extraction (Frozen AlexNet)")
print("=" * 50)

model_feature_extract = models.alexnet(pretrained=True)

# Freeze all layers except the final classifier layer
for param in model_feature_extract.features.parameters():
    param.requires_grad = False

# Only the newly added layer will be trainable
model_feature_extract.classifier[6] = nn.Linear(4096, 10)
model_feature_extract = model_feature_extract.to(device)

# Only optimize the new classifier layer
optimizer_feature_extract = torch.optim.Adam(model_feature_extract.classifier[6].parameters(), lr=learning_rate)

# Training loop for feature extraction
best_val_loss_fe = float('inf')
for epoch in range(epochs):
    model_feature_extract.train()
    running_loss = 0.0
    
    for images, labels in trainloader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer_feature_extract.zero_grad()
        
        outputs = model_feature_extract(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer_feature_extract.step()
        
        running_loss += loss.item()
    
    avg_train_loss = running_loss / len(trainloader)
    
    # Validation
    model_feature_extract.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in valloader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model_feature_extract(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(valloader)
    
    print(f"Epoch [{epoch+1}/{epochs}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    
    # Save best model
    if avg_val_loss < best_val_loss_fe:
        best_val_loss_fe = avg_val_loss
        torch.save(model_feature_extract.state_dict(), "BestModel_FeatureExtract.pth")

# Test the feature extraction model
model_feature_extract.load_state_dict(torch.load("BestModel_FeatureExtract.pth"))
model_feature_extract.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model_feature_extract(images)
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy_feature_extract = 100 * (correct / total)
print(f"\nTest Accuracy (Feature Extraction): {accuracy_feature_extract:.2f}%")
wandb.log({"feature_extract_test_accuracy": accuracy_feature_extract})

# ======================== COMPARISON ========================
print("\n" + "=" * 50)
print("COMPARISON SUMMARY")
print("=" * 50)
print(f"Fine-Tuning Test Accuracy:      {accuracy_finetune:.2f}%")
print(f"Feature Extraction Accuracy:    {accuracy_feature_extract:.2f}%")
print(f"Difference:                     {abs(accuracy_finetune - accuracy_feature_extract):.2f}%")
print("\nExplanation:")
print("- Fine-tuning allows all layers to be updated with CIFAR-10 data,")
print("  adapting the learned ImageNet features to the new task.")
print("- Feature extraction keeps the pre-trained weights frozen,")
print("  using ImageNet features as fixed representations.")
print("- Fine-tuning typically performs better as it can specialize")
print("  the model filters to the specific CIFAR-10 task.")
wandb.finish()

Task 0.2.2

Prepare a CNN of your choice and train it on the MNIST data. Report the accuracy

In [15]:
#Task 2.2
traindata_MNIST = torchvision.datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transformer_MNIST)

testdata_MNIST = torchvision.datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transformer_MNIST)


train_size = int(0.8*len(traindata_MNIST))
val_size = len(traindata_MNIST) - train_size

train_dataset_MNIST, val_dataset_MNIST = random_split(traindata_MNIST, [train_size, val_size], generator=torch.Generator().manual_seed(42))

trainload_MNIST = torch.utils.data.DataLoader(traindata_MNIST, batch_size=32, shuffle=True)
valload_MNIST = torch.utils.data.DataLoader(val_dataset_MNIST, batch_size=32, shuffle=False)
testload_MNIST = torch.utils.data.DataLoader(testdata_MNIST, batch_size=32, shuffle=False)

model = CNN_MNIST(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.0001)
num_epochs = 10

In [16]:
best_val_loss = float('inf')

for epoch in range(num_epochs):
    # ------------------------- TRAIN -------------------------
    model.train()
    running_loss = 0.0

    for images, labels in trainload_MNIST:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(trainload_MNIST)

    # ------------------------- VALIDATION -------------------------

    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in valload_MNIST:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()

        #     wandb.log({
        #     "validation loss": val_running_loss
        # })

        average_val_loss = val_running_loss / len(valload_MNIST)

        # wandb.log({
        #     "average validation loss": average_val_loss
        # })

    # scheduler.step(average_loss)

    # ------------------------- Saving Best Model -------------------------
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        torch.save(model.state_dict(), "BestModel_MNIST.pth")
        print("Saved The Best Performing Model")

    # ------------------------- Visualization -------------------------
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"train loss: {average_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )

Saved The Best Performing Model
Epoch [1/10] train loss: 1.846, val loss: 1.410, lr: 0.000100
Saved The Best Performing Model
Epoch [2/10] train loss: 1.143, val loss: 0.897, lr: 0.000100
Saved The Best Performing Model
Epoch [3/10] train loss: 0.779, val loss: 0.648, lr: 0.000100
Saved The Best Performing Model
Epoch [4/10] train loss: 0.595, val loss: 0.510, lr: 0.000100
Saved The Best Performing Model
Epoch [5/10] train loss: 0.486, val loss: 0.430, lr: 0.000100
Saved The Best Performing Model
Epoch [6/10] train loss: 0.419, val loss: 0.374, lr: 0.000100
Saved The Best Performing Model
Epoch [7/10] train loss: 0.370, val loss: 0.329, lr: 0.000100
Saved The Best Performing Model
Epoch [8/10] train loss: 0.334, val loss: 0.300, lr: 0.000100
Saved The Best Performing Model
Epoch [9/10] train loss: 0.307, val loss: 0.274, lr: 0.000100
Saved The Best Performing Model
Epoch [10/10] train loss: 0.285, val loss: 0.258, lr: 0.000100


In [17]:
#Test best model for MNIST
model.load_state_dict(torch.load("BestModel_MNIST.pth"))
model.to(device)
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload_MNIST:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # wandb.log({
        #     "test loss" : test_loss

        # })          

average_test_loss = test_loss / len(testload_MNIST)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")

# wandb.finish()


Test loss: 0.235
Test accuracy: 94.65%


Use the above model as a pre-trained CNN for the SVHN dataset. Report the accuracy

In [18]:
#Import SVHN Dataset 0.2.2
traindata_SVHN = torchvision.datasets.SVHN(
    root='./data',
    split="train",
    download=True,
    transform=transformer_SVHN)

testdata_SVHN = torchvision.datasets.SVHN(
    root='./data',
    split="test",
    download=True,
    transform=transformer_SVHN)


train_size = int(0.8*len(traindata_SVHN))
val_size = len(traindata_SVHN) - train_size

train_dataset_SVHN, val_dataset_SVHN = random_split(traindata_SVHN, [train_size, val_size], generator=torch.Generator().manual_seed(42))

trainload_SVHN = torch.utils.data.DataLoader(traindata_SVHN, batch_size=32, shuffle=True)
valload_SVHN = torch.utils.data.DataLoader(val_dataset_SVHN, batch_size=32, shuffle=False)
testload_SVHN = torch.utils.data.DataLoader(testdata_SVHN, batch_size=32, shuffle=False)

100.0%
100.0%


In [19]:
#Test MNIST model on SVHN dataset

#Test best model for MNIST
model.load_state_dict(torch.load("BestModel_MNIST.pth"))
model.to(device)
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload_SVHN:
        images = images.to(device)
        if images.shape[1] == 3:  # RGB
            images = images.mean(dim=1, keepdim=True)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # wandb.log({
        #     "test loss" : test_loss

        # })

average_test_loss = test_loss / len(testload_SVHN)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")

# wandb.finish()


Test loss: 2.329
Test accuracy: 14.99%


In the third step you are performing transfer learning from MNIST to SVHN.

In [20]:
#Transfer learning from MNIST to SVHN 
model.load_state_dict(torch.load("BestModel_MNIST.pth"))
model.to(device)

#Freeze model weights
for param in model.parameters():
    param.requires_grad = False

model.classifier[4] = nn.Linear(128, 10)
model = model.to(device) 
optimizer = torch.optim.SGD(model.classifier[4].parameters(), lr=0.0001)

best_val_loss = float('inf')

for epoch in range(num_epochs):
    # ------------------------- TRAIN -------------------------
    model.train()
    running_loss = 0.0

    for images, labels in trainload_SVHN:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(trainload_SVHN)

    # ------------------------- VALIDATION -------------------------

    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for images, labels in valload_SVHN:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            val_loss = criterion(outputs, labels)
            val_running_loss += val_loss.item()

        #     wandb.log({
        #     "validation loss": val_running_loss
        # })

        average_val_loss = val_running_loss / len(valload_SVHN)

        # wandb.log({
        #     "average validation loss": average_val_loss
        # })

    # scheduler.step(average_loss)

    # ------------------------- Saving Best Model -------------------------
    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        torch.save(model.state_dict(), "BestModel_SVHN_Transfer.pth")
        print("Saved The Best Performing Model")

    # ------------------------- Visualization -------------------------
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"train loss: {average_loss:.3f}, "
        f"val loss: {average_val_loss:.3f}, "
        f"lr: {optimizer.param_groups[0]['lr']:.6f}"
    )

Saved The Best Performing Model
Epoch [1/10] train loss: 2.307, val loss: 2.246, lr: 0.000100
Saved The Best Performing Model
Epoch [2/10] train loss: 2.258, val loss: 2.221, lr: 0.000100
Saved The Best Performing Model
Epoch [3/10] train loss: 2.235, val loss: 2.202, lr: 0.000100
Saved The Best Performing Model
Epoch [4/10] train loss: 2.215, val loss: 2.184, lr: 0.000100
Saved The Best Performing Model
Epoch [5/10] train loss: 2.197, val loss: 2.170, lr: 0.000100
Saved The Best Performing Model
Epoch [6/10] train loss: 2.184, val loss: 2.153, lr: 0.000100
Saved The Best Performing Model
Epoch [7/10] train loss: 2.169, val loss: 2.140, lr: 0.000100
Saved The Best Performing Model
Epoch [8/10] train loss: 2.158, val loss: 2.129, lr: 0.000100
Saved The Best Performing Model
Epoch [9/10] train loss: 2.148, val loss: 2.119, lr: 0.000100
Saved The Best Performing Model
Epoch [10/10] train loss: 2.139, val loss: 2.111, lr: 0.000100


In [21]:
#Test transfer model on SVHN dataset

#Test best model for MNIST
model.load_state_dict(torch.load("BestModel_SVHN_Transfer.pth"))
model.to(device)
model.eval()

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testload_SVHN:
        images = images.to(device)
        if images.shape[1] == 3:  # RGB
            images = images.mean(dim=1, keepdim=True)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        HighestValue, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        # wandb.log({
        #     "test loss" : test_loss

        # })

average_test_loss = test_loss / len(testload_SVHN)
test_acc = 100*(correct/total)


print(f"Test loss: {average_test_loss:.3f}")
print(f"Test accuracy: {test_acc:.2f}%")

# wandb.finish()

Test loss: 2.099
Test accuracy: 27.58%
